In [1]:
# Copyright (c) 2026 Rioborue Alexander Oghenerume. All rights reserved.
# This code cannot be used, modified, or distributed without permission.
!pip install pingouin scikit-learn pandas numpy scipy openpyxl -q

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, skew
import pingouin as pg
from sklearn.metrics import cohen_kappa_score
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("RELIABILITY ANALYSIS ENVIRONMENT INITIALIZED")
print("="*70)
print("Libraries loaded successfully:")
print("  • pandas (data manipulation)")
print("  • numpy (numerical operations)")
print("  • scipy (statistical tests)")
print("  • pingouin (ICC calculation)")
print("  • sklearn (Cohen's Kappa)")
print("  • openpyxl (Excel file support)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 5.1 MB/s eta 0:00:00
RELIABILITY ANALYSIS ENVIRONMENT INITIALIZED
Libraries loaded successfully:
  • pandas (data manipulation)
  • numpy (numerical operations)
  • scipy (statistical tests)
  • pingouin (ICC calculation)
  • sklearn (Cohen's Kappa)
  • openpyxl (Excel file support)


In [2]:
print("\n" + "="*70)
print("DATA IMPORT")
print("="*70)
print("Please upload your Excel file (.xlsx) containing the assessment data.")
print("Expected structure: MCQ columns (Q1-Q20) and Essay columns")

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename, engine='openpyxl')

print(f"\n✓ Data successfully loaded: {filename}")
print(f"  • Number of respondents: {len(df)}")
print(f"  • Number of variables: {len(df.columns)}")
print(f"  • Variable names: {df.columns.tolist()}")

print("\nFirst 5 rows of data:")
print(df.head())


DATA IMPORT
Please upload your Excel file (.xlsx) containing the assessment data.
Expected structure: MCQ columns (Q1-Q20) and Essay columns


Saving DP.xlsx to DP.xlsx

✓ Data successfully loaded: DP.xlsx
  • Number of respondents: 20
  • Number of variables: 27
  • Variable names: ['Student_ID', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Essay1_Rater1', 'Essay1_Rater2', 'Essay2_Rater1', 'Essay2_Rater2', 'Essay3_Rater1', 'Essay3_Rater2']

First 5 rows of data:
  Student_ID  Q1  Q2  Q3  Q4  Q5  Q6  Q7  Q8  Q9  ...  Q17  Q18  Q19  Q20  \
0         S1   1   0   1   1   1   1   1   0   1  ...    1    1    1    1   
1         S2   1   1   1   1   1   1   1   1   1  ...    1    1    1    1   
2         S3   1   1   1   1   1   1   1   1   1  ...    1    1    1    1   
3         S4   1   1   1   1   1   1   1   1   1  ...    0    1    1    1   
4         S5   1   1   1   1   1   1   1   1   0  ...    1    1    1    1   

   Essay1_Rater1  Essay1_Rater2  Essay2_Rater1  Essay2_Rater2  Essay3_Rater1  \
0             14             16             1

In [3]:
print("\n" + "="*70)
print("DATA VALIDATION")
print("="*70)

mcq_columns = [f'Q{i}' for i in range(1, 21)]
missing_mcq = [col for col in mcq_columns if col not in df.columns]

if missing_mcq:
    print(f"Warning: Missing MCQ columns: {missing_mcq}")
    print(f"Available columns starting with 'Q': {[col for col in df.columns if col.startswith('Q')]}")
else:
    print("✓ All 20 MCQ columns (Q1-Q20) present")

essay_columns = ['Essay1_Rater1', 'Essay1_Rater2', 'Essay2_Rater1',
                 'Essay2_Rater2', 'Essay3_Rater1', 'Essay3_Rater2']
missing_essay = [col for col in essay_columns if col not in df.columns]

if missing_essay:
    print(f"Warning: Missing essay columns: {missing_essay}")
else:
    print("✓ All 6 essay columns present")


DATA VALIDATION
✓ All 20 MCQ columns (Q1-Q20) present
✓ All 6 essay columns present


In [4]:
def calculate_kr20(item_data):
    """
    Calculate Kuder-Richardson Formula 20 (KR-20) for dichotomous items.

    Parameters
    ----------
    item_data : pandas.DataFrame
        DataFrame containing dichotomous items (scored 0 or 1)

    Returns
    -------
    tuple
        (kr20_coefficient, total_scores, item_variances, total_variance)
    """
    total_scores = item_data.sum(axis=1)
    total_variance = total_scores.var(ddof=1)
    item_variances = item_data.var(ddof=0)
    sum_item_variances = item_variances.sum()
    k = item_data.shape[1]
    kr20 = (k / (k - 1)) * (1 - (sum_item_variances / total_variance))
    return kr20, total_scores, item_variances, total_variance

def interpret_reliability(coefficient, metric_name, thresholds=None):
    """
    Provide professional interpretation of reliability coefficients.

    Parameters
    ----------
    coefficient : float
        Reliability coefficient value
    metric_name : str
        Name of the metric (for reporting)
    thresholds : dict, optional
        Custom thresholds for interpretation

    Returns
    -------
    str
        Professional interpretation
    """
    if thresholds is None:
        thresholds = {
            'excellent': 0.90,
            'good': 0.80,
            'moderate': 0.70,
            'questionable': 0.60
        }

    if coefficient >= thresholds['excellent']:
        return f"Excellent {metric_name} (suitable for high-stakes decisions)"
    elif coefficient >= thresholds['good']:
        return f"Good {metric_name} (acceptable for most purposes)"
    elif coefficient >= thresholds['moderate']:
        return f"Moderate {metric_name} (adequate for research purposes)"
    elif coefficient >= thresholds['questionable']:
        return f"Questionable {metric_name} (requires item review)"
    else:
        return f"Poor {metric_name} (requires substantial revision)"

In [5]:
print("\n" + "="*70)
print("SECTION A: MULTIPLE CHOICE RELIABILITY (KR-20)")
print("="*70)

mcq_items = df[mcq_columns]
kr20_score, sectionA_total, item_variances, total_var = calculate_kr20(mcq_items)

print(f"\nKR-20 Internal Consistency Coefficient")
print("-"*50)
print(f"Coefficient:           {kr20_score:.4f}")
print(f"Number of Items:       20")
print(f"Sample Size:           {len(df)}")
print(f"Interpretation:        {interpret_reliability(kr20_score, 'internal consistency')}")

se_kr20 = np.sqrt((1 - kr20_score**2) / (len(df) - 1))
print(f"Standard Error:        ±{se_kr20:.4f}")

print(f"\nDiagnostic Statistics")
print("-"*50)
print(f"Mean MCQ Score:        {sectionA_total.mean():.2f} out of 20")
print(f"Median MCQ Score:      {sectionA_total.median():.0f}")
print(f"Standard Deviation:    {sectionA_total.std(ddof=1):.2f}")
print(f"Score Range:           {sectionA_total.min():.0f} - {sectionA_total.max():.0f}")
print(f"Total Variance:        {total_var:.4f}")
print(f"Sum of Item Variances: {item_variances.sum():.4f}")

print(f"\nKR-20 Formula Breakdown")
print("-"*50)
print(f"KR-20 = (k/(k-1)) × (1 - (Σσ²_i / σ²_total))")
print(f"      = (20/19) × (1 - ({item_variances.sum():.4f} / {total_var:.4f}))")
print(f"      = 1.0526 × (1 - {item_variances.sum()/total_var:.4f})")
print(f"      = {kr20_score:.4f}")


SECTION A: MULTIPLE CHOICE RELIABILITY (KR-20)

KR-20 Internal Consistency Coefficient
--------------------------------------------------
Coefficient:           0.8137
Number of Items:       20
Sample Size:           20
Interpretation:        Good internal consistency (acceptable for most purposes)
Standard Error:        ±0.1334

Diagnostic Statistics
--------------------------------------------------
Mean MCQ Score:        11.05 out of 20
Median MCQ Score:      11
Standard Deviation:    4.57
Score Range:           3 - 19
Total Variance:        20.8921
Sum of Item Variances: 4.7425

KR-20 Formula Breakdown
--------------------------------------------------
KR-20 = (k/(k-1)) × (1 - (Σσ²_i / σ²_total))
      = (20/19) × (1 - (4.7425 / 20.8921))
      = 1.0526 × (1 - 0.2270)
      = 0.8137


In [6]:
print("\n" + "="*70)
print("ITEM DIFFICULTY AND DISCRIMINATION ANALYSIS")
print("="*70)

print("\nItem Difficulty (p-values) and Item-Total Correlations")
print("-"*70)
print("Item | Difficulty (p) | Classification | Item-Total r | Quality")
print("-"*70)

easy_count = 0
hard_count = 0
good_discrimination = 0

for i, col in enumerate(mcq_columns, 1):
    p_value = mcq_items[col].mean()
    item_total_corr = mcq_items[col].corr(sectionA_total)

    if np.isnan(item_total_corr):
        item_total_corr = 0

    if p_value >= 0.80:
        difficulty_class = "Easy"
        easy_count += 1
    elif p_value <= 0.20:
        difficulty_class = "Difficult"
        hard_count += 1
    else:
        difficulty_class = "Moderate"

    # Discrimination quality
    if item_total_corr >= 0.20:
        quality = "Good"
        good_discrimination += 1
    else:
        quality = "Review"

    print(f"Q{i:2d}  | {p_value:.3f}       | {difficulty_class:9s} | {item_total_corr:.3f}       | {quality:6s}")

print("-"*70)

print(f"\nItem Difficulty Distribution")
print("-"*50)
print(f"  • Easy items (p ≥ 0.80):        {easy_count}")
print(f"  • Moderate items (0.20 < p < 0.80): {20 - easy_count - hard_count}")
print(f"  • Difficult items (p ≤ 0.20):    {hard_count}")

print(f"\nItem Discrimination Quality")
print("-"*50)
print(f"  • Good discrimination (r ≥ 0.20): {good_discrimination}")
print(f"  • Needs review (r < 0.20):        {20 - good_discrimination}")


ITEM DIFFICULTY AND DISCRIMINATION ANALYSIS

Item Difficulty (p-values) and Item-Total Correlations
----------------------------------------------------------------------
Item | Difficulty (p) | Classification | Item-Total r | Quality
----------------------------------------------------------------------
Q 1  | 0.650       | Moderate  | 0.149       | Review
Q 2  | 0.550       | Moderate  | 0.642       | Good  
Q 3  | 0.700       | Moderate  | 0.644       | Good  
Q 4  | 0.700       | Moderate  | 0.595       | Good  
Q 5  | 0.700       | Moderate  | 0.644       | Good  
Q 6  | 0.550       | Moderate  | 0.800       | Good  
Q 7  | 0.550       | Moderate  | 0.529       | Good  
Q 8  | 0.500       | Moderate  | 0.415       | Good  
Q 9  | 0.650       | Moderate  | 0.408       | Good  
Q10  | 0.650       | Moderate  | 0.126       | Review
Q11  | 0.500       | Moderate  | 0.438       | Good  
Q12  | 0.600       | Moderate  | 0.490       | Good  
Q13  | 0.400       | Moderate  | 0.243       

In [7]:
print("\n" + "="*70)
print("SECTION B: ESSAY INTER-RATER RELIABILITY")
print("="*70)

rater1_avg = df[['Essay1_Rater1', 'Essay2_Rater1', 'Essay3_Rater1']].mean(axis=1)
rater2_avg = df[['Essay1_Rater2', 'Essay2_Rater2', 'Essay3_Rater2']].mean(axis=1)

corr, p_value = pearsonr(rater1_avg, rater2_avg)

print(f"\nRater Agreement (Pearson Correlation)")
print("-"*50)
print(f"Correlation Coefficient: {corr:.4f}")
print(f"p-value:                 {p_value:.4f}")
print(f"Significance:            {'Statistically significant' if p_value < 0.05 else 'Not statistically significant'}")

if corr > 0.80:
    correlation_strength = "Strong positive correlation"
elif corr > 0.50:
    correlation_strength = "Moderate positive correlation"
else:
    correlation_strength = "Weak correlation"
print(f"Interpretation:          {correlation_strength}")

rater_data = pd.DataFrame({
    'Student': df['Student_ID'].tolist() * 2,
    'Rater': ['Rater1'] * len(df) + ['Rater2'] * len(df),
    'Score': rater1_avg.tolist() + rater2_avg.tolist()
})

icc_result = pg.intraclass_corr(data=rater_data, targets='Student', raters='Rater', ratings='Score')

print(f"\nIntraclass Correlation Coefficient (ICC)")
print("-"*50)
print(icc_result.to_string())

try:
    icc_value = icc_result[icc_result['Type'] == 'ICC2k']['ICC'].values[0]
    icc_type = "ICC2k (average of raters)"
    icc_ci_lower = icc_result[icc_result['Type'] == 'ICC2k']['CI95%'].values[0][0]
    icc_ci_upper = icc_result[icc_result['Type'] == 'ICC2k']['CI95%'].values[0][1]
except:
    try:
        icc_value = icc_result[icc_result['Type'] == 'ICC2']['ICC'].values[0]
        icc_type = "ICC2 (single rater)"
        icc_ci_lower = icc_result[icc_result['Type'] == 'ICC2']['CI95%'].values[0][0]
        icc_ci_upper = icc_result[icc_result['Type'] == 'ICC2']['CI95%'].values[0][1]
    except:
        icc_value = icc_result['ICC'].values[0]
        icc_type = "ICC (first available)"
        icc_ci_lower = None
        icc_ci_upper = None

print(f"\nICC Coefficient Summary")
print("-"*50)
print(f"ICC Value:          {icc_value:.4f}")
print(f"ICC Type:           {icc_type}")
if icc_ci_lower and icc_ci_upper:
    print(f"95% CI:            [{icc_ci_lower:.4f}, {icc_ci_upper:.4f}]")
print(f"Interpretation:     {interpret_reliability(icc_value, 'inter-rater agreement')}")

print(f"\nCohen's Kappa for Individual Essays")
print("-"*50)
essay_pairs = [
    ('Essay1_Rater1', 'Essay1_Rater2'),
    ('Essay2_Rater1', 'Essay2_Rater2'),
    ('Essay3_Rater1', 'Essay3_Rater2')
]
kappa_values = []
kappa_labels = []

for e1, e2 in essay_pairs:
    kappa = cohen_kappa_score(np.round(df[e1]), np.round(df[e2]))
    kappa_values.append(kappa)
    kappa_labels.append(f"{e1} vs {e2}")

    if kappa >= 0.81:
        k_interp = "Almost perfect"
    elif kappa >= 0.61:
        k_interp = "Substantial"
    elif kappa >= 0.41:
        k_interp = "Moderate"
    elif kappa >= 0.21:
        k_interp = "Fair"
    else:
        k_interp = "Poor"

    print(f"{e1:20s} vs {e2:20s}: {kappa:.4f} ({k_interp})")


SECTION B: ESSAY INTER-RATER RELIABILITY

Rater Agreement (Pearson Correlation)
--------------------------------------------------
Correlation Coefficient: 0.9501
p-value:                 0.0000
Significance:            Statistically significant
Interpretation:          Strong positive correlation

Intraclass Correlation Coefficient (ICC)
--------------------------------------------------
       Type       ICC          F  df1  df2          pval          CI95
0  ICC(1,1)  0.951178  39.965256   19   20  6.492916e-12  [0.88, 0.98]
1  ICC(A,1)  0.951149  38.999329   19   19  2.352343e-11  [0.88, 0.98]
2  ICC(C,1)  0.949999  38.999329   19   19  2.352343e-11  [0.88, 0.98]
3  ICC(1,k)  0.974978  39.965256   19   20  6.492916e-12  [0.94, 0.99]
4  ICC(A,k)  0.974963  38.999329   19   19  2.352343e-11  [0.94, 0.99]
5  ICC(C,k)  0.974359  38.999329   19   19  2.352343e-11  [0.94, 0.99]

ICC Coefficient Summary
--------------------------------------------------
ICC Value:          0.9512
ICC Typ

In [8]:
print("\n" + "="*70)
print("SECTION A vs SECTION B: CONSTRUCT RELATIONSHIP")
print("="*70)

essay1_avg = (df['Essay1_Rater1'] + df['Essay1_Rater2']) / 2
essay2_avg = (df['Essay2_Rater1'] + df['Essay2_Rater2']) / 2
essay3_avg = (df['Essay3_Rater1'] + df['Essay3_Rater2']) / 2
sectionB_total = essay1_avg + essay2_avg + essay3_avg
overall_total = sectionA_total + sectionB_total

corr_AB, p_AB = pearsonr(sectionA_total, sectionB_total)

print(f"\nCorrelation between MCQ and Essay Sections")
print("-"*50)
print(f"Pearson Correlation: {corr_AB:.4f}")
print(f"p-value:             {p_AB:.4f}")
print(f"Recommended Range:   0.40 - 0.60 for achievement tests")

if 0.40 <= corr_AB <= 0.60:
    relationship_interpretation = "Optimal - Sections measure related but distinct constructs"
    print("✓ GOOD - Sections measure related but distinct constructs")
elif corr_AB < 0.30:
    relationship_interpretation = "Weak - Sections may measure different constructs"
    print("⚠ WEAK - Sections may measure completely different constructs")
elif corr_AB < 0.70:
    relationship_interpretation = "Moderate - Some relationship but not ideal"
    print("⚠ MODERATE - Some relationship but not ideal")
else:
    relationship_interpretation = "Strong - Sections may be redundant"
    print("⚠ STRONG - Sections may be redundant")


SECTION A vs SECTION B: CONSTRUCT RELATIONSHIP

Correlation between MCQ and Essay Sections
--------------------------------------------------
Pearson Correlation: 0.8409
p-value:             0.0000
Recommended Range:   0.40 - 0.60 for achievement tests
⚠ STRONG - Sections may be redundant


In [9]:
print("\n" + "="*70)
print("DESCRIPTIVE STATISTICS")
print("="*70)

print(f"\nSECTION A (Multiple Choice Questions)")
print("-"*50)
print(f"  Maximum Possible: 20 marks")
print(f"  Mean:             {sectionA_total.mean():.2f}")
print(f"  Median:           {sectionA_total.median():.0f}")
print(f"  Standard Dev:     {sectionA_total.std(ddof=1):.2f}")
print(f"  Minimum:          {sectionA_total.min():.0f}")
print(f"  Maximum:          {sectionA_total.max():.0f}")
print(f"  Skewness:         {sectionA_total.skew():.3f}")

print(f"\nSECTION B (Essay Questions)")
print("-"*50)
print(f"  Maximum Possible: 50 marks")
print(f"  Mean:             {sectionB_total.mean():.2f}")
print(f"  Median:           {sectionB_total.median():.1f}")
print(f"  Standard Dev:     {sectionB_total.std(ddof=1):.2f}")
print(f"  Minimum:          {sectionB_total.min():.0f}")
print(f"  Maximum:          {sectionB_total.max():.0f}")
print(f"  Skewness:         {sectionB_total.skew():.3f}")

print(f"\nOVERALL TEST")
print("-"*50)
print(f"  Maximum Possible: 70 marks")
print(f"  Mean:             {overall_total.mean():.2f}")
print(f"  Median:           {overall_total.median():.1f}")
print(f"  Standard Dev:     {overall_total.std(ddof=1):.2f}")
print(f"  Minimum:          {overall_total.min():.0f}")
print(f"  Maximum:          {overall_total.max():.0f}")
print(f"  Skewness:         {overall_total.skew():.3f}")


DESCRIPTIVE STATISTICS

SECTION A (Multiple Choice Questions)
--------------------------------------------------
  Maximum Possible: 20 marks
  Mean:             11.05
  Median:           11
  Standard Dev:     4.57
  Minimum:          3
  Maximum:          19
  Skewness:         0.034

SECTION B (Essay Questions)
--------------------------------------------------
  Maximum Possible: 50 marks
  Mean:             29.02
  Median:           29.5
  Standard Dev:     8.74
  Minimum:          14
  Maximum:          43
  Skewness:         -0.286

OVERALL TEST
--------------------------------------------------
  Maximum Possible: 70 marks
  Mean:             40.08
  Median:           41.0
  Standard Dev:     12.83
  Minimum:          20
  Maximum:          62
  Skewness:         -0.053


In [10]:
print("\n" + "="*70)
print("COMPREHENSIVE RELIABILITY SUMMARY")
print("="*70)

summary_data = {
    'Analysis': [
        'KR-20 (Multiple Choice)',
        'ICC (Essay Inter-Rater)',
        'Section A-B Correlation'
    ],
    'Coefficient': [kr20_score, icc_value, corr_AB],
    '95% CI Lower': [
        kr20_score - 1.96 * se_kr20,
        icc_ci_lower if icc_ci_lower else np.nan,
        np.nan
    ],
    '95% CI Upper': [
        kr20_score + 1.96 * se_kr20,
        icc_ci_upper if icc_ci_upper else np.nan,
        np.nan
    ],
    'Benchmark': [
        '> 0.80 recommended',
        '> 0.75 recommended',
        '0.40 - 0.60 ideal'
    ],
    'Interpretation': [
        interpret_reliability(kr20_score, 'internal consistency'),
        interpret_reliability(icc_value, 'inter-rater agreement'),
        relationship_interpretation
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.round(4).to_string(index=False))


COMPREHENSIVE RELIABILITY SUMMARY

               Analysis  Coefficient  95% CI Lower  95% CI Upper          Benchmark                                                       Interpretation
KR-20 (Multiple Choice)       0.8137        0.5523        1.0751 > 0.80 recommended             Good internal consistency (acceptable for most purposes)
ICC (Essay Inter-Rater)       0.9512           NaN           NaN > 0.75 recommended Excellent inter-rater agreement (suitable for high-stakes decisions)
Section A-B Correlation       0.8409           NaN           NaN  0.40 - 0.60 ideal                                   Strong - Sections may be redundant


In [11]:
print("\n" + "="*70)
print("RECOMMENDATIONS")
print("="*70)

issues = []

if kr20_score < 0.70:
    issues.append({
        'area': 'Internal Consistency (KR-20)',
        'issue': f'KR-20 = {kr20_score:.3f} (below recommended threshold)',
        'recommendation': 'Review items with low item-total correlations. Consider increasing test length or revising weak items.'
    })

if icc_value < 0.75:
    issues.append({
        'area': 'Inter-Rater Agreement (ICC)',
        'issue': f'ICC = {icc_value:.3f} (below recommended threshold)',
        'recommendation': 'Implement rater training sessions. Refine scoring rubric with specific criteria and anchors.'
    })

if corr_AB < 0.40:
    issues.append({
        'area': 'Test Structure (Sections Relationship)',
        'issue': f'Correlation = {corr_AB:.3f} (below ideal range)',
        'recommendation': 'Review assessment blueprint. Ensure sections align with course objectives and measure related constructs.'
    })
elif corr_AB > 0.60:
    issues.append({
        'area': 'Test Structure (Sections Relationship)',
        'issue': f'Correlation = {corr_AB:.3f} (above ideal range)',
        'recommendation': 'Consider reducing redundancy between sections. Sections may be measuring the same construct.'
    })

if issues:
    print("\nThe following areas require attention:\n")
    for i, issue in enumerate(issues, 1):
        print(f"{i}. {issue['area']}")
        print(f"   Issue: {issue['issue']}")
        print(f"   Action: {issue['recommendation']}\n")
else:
    print("\nAll reliability metrics are within acceptable ranges.")
    print("The test demonstrates strong psychometric properties.")
    print("Maintain current assessment structure as a benchmark for future administrations.")

print("\n" + "="*70)
print("DATA EXPORT")
print("="*70)

results = {
    'Metric': [
        'KR-20', 'ICC', 'Section A-B Correlation',
        'Section A Mean', 'Section A SD', 'Section A Skewness',
        'Section B Mean', 'Section B SD', 'Section B Skewness',
        'Overall Mean', 'Overall SD', 'Overall Skewness'
    ],
    'Value': [
        kr20_score, icc_value, corr_AB,
        sectionA_total.mean(), sectionA_total.std(ddof=1), sectionA_total.skew(),
        sectionB_total.mean(), sectionB_total.std(ddof=1), sectionB_total.skew(),
        overall_total.mean(), overall_total.std(ddof=1), overall_total.skew()
    ]
}

results_df = pd.DataFrame(results)
results_filename = 'reliability_analysis_results.csv'
results_df.to_csv(results_filename, index=False)
print(f"\n✓ Results exported to: {results_filename}")

files.download(results_filename)
print(f"✓ Results downloaded to your computer")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)
print(f"\n  Analyst:          R.A. Oghenerume")
print(f"  Analysis Date:    {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print(f"  Analysis Time:    {pd.Timestamp.now().strftime('%H:%M:%S')}")
print(f"  Sample Size:      {len(df)} respondents")
print(f"  Methods:          KR-20, ICC(2,k), Cohen's Kappa, Pearson correlation")
print(f"  Output File:      {results_filename}")
print("\n" + "="*70)
print("© R.A. Oghenerume. All Rights Reserved.")
print("="*70)


RECOMMENDATIONS

The following areas require attention:

1. Test Structure (Sections Relationship)
   Issue: Correlation = 0.841 (above ideal range)
   Action: Consider reducing redundancy between sections. Sections may be measuring the same construct.


DATA EXPORT

✓ Results exported to: reliability_analysis_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Results downloaded to your computer

ANALYSIS COMPLETE

  Analyst:          R.A. Oghenerume
  Analysis Date:    2026-06-19
  Analysis Time:    20:12:13
  Sample Size:      20 respondents
  Methods:          KR-20, ICC(2,k), Cohen's Kappa, Pearson correlation
  Output File:      reliability_analysis_results.csv

© R.A. Oghenerume. All Rights Reserved.
